In [1]:
import sys
import os
import requests
from datetime import datetime, timedelta
from fastapi import FastAPI, Query
from fastapi.testclient import TestClient

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from shared.config import Config

 Using model: deepseek/deepseek-v3.2


In [2]:
def get_weather(location: str, period: str = "current", dt: str = None):
    api_key = Config.Weather.OPENWEATHER_API_KEY
    base_url = "http://api.weatherapi.com/v1"
    
    if period == "forecast":
        url = f"{base_url}/forecast.json?key={api_key}&q={location}&days=7&lang=th"
    elif period == "historical":
        if not dt:
            dt = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
        url = f"{base_url}/history.json?key={api_key}&q={location}&dt={dt}&lang=th"
    else:
        url = f"{base_url}/current.json?key={api_key}&q={location}&lang=th"

    try:
        response = requests.get(url)
        data = response.json()
        
        if "error" in data:
            return {"error": data["error"]["message"]}
            
        return {
            "display_name": f"{data.get('location', {}).get('name', 'Unknown')}, {data.get('location', {}).get('country', 'Unknown')}",
            
            "temp": data.get("current", {}).get("temp_c") if period == "current" else f"See {period} data",
            "condition": data.get("current", {}).get("condition", {}).get("text") if period == "current" else f"See {period} data",
            
            "full_data": data
        }
    except Exception as e:
        return {"error": str(e)}

In [3]:
app = FastAPI()

@app.get("/weather")
def weather_endpoint(
    location: str, 
    period: str = Query("current", enum=["current", "forecast", "historical"]),
    dt: str = Query(None, description="Date in YYYY-MM-DD format for historical data")
):
    return get_weather(location, period, dt)


client = TestClient(app)

In [4]:
print("Current Weather")
res_current = client.get("/weather", params={"location": "Bangkok"})
print(res_current.json(), "\n")

print("2. Forecast")
res_forecast = client.get("/weather", params={"location": "Phuket", "period": "forecast"})
print(res_forecast.json(), "\n")

print(" 3. Historical")
res_history = client.get("/weather", params={"location": "Chiang Mai", "period": "historical", "dt": "2024-03-20"})
print(res_history.json())


Current Weather
{'display_name': 'Bangkok, Thailand', 'temp': 37.3, 'condition': 'Sunny', 'full_data': {'location': {'name': 'Bangkok', 'region': 'Krung Thep', 'country': 'Thailand', 'lat': 13.75, 'lon': 100.5167, 'tz_id': 'Asia/Bangkok', 'localtime_epoch': 1775806331, 'localtime': '2026-04-10 14:32'}, 'current': {'last_updated_epoch': 1775806200, 'last_updated': '2026-04-10 14:30', 'temp_c': 37.3, 'temp_f': 99.1, 'is_day': 1, 'condition': {'text': 'Sunny', 'icon': '//cdn.weatherapi.com/weather/64x64/day/113.png', 'code': 1000}, 'wind_mph': 12.3, 'wind_kph': 19.8, 'wind_degree': 178, 'wind_dir': 'S', 'pressure_mb': 1009.0, 'pressure_in': 29.8, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 48, 'cloud': 25, 'feelslike_c': 38.2, 'feelslike_f': 100.8, 'windchill_c': 37.9, 'windchill_f': 100.2, 'heatindex_c': 39.2, 'heatindex_f': 102.5, 'dewpoint_c': 15.9, 'dewpoint_f': 60.7, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 10.1, 'gust_mph': 14.2, 'gust_kph': 22.8, 'short_rad': 987.72, 'diff_rad':

In [ ]:
response_jp = requests.get("/weather", params={"location": "Japan", "period": "current"})
response_jp